In [ ]:
from py123d.api import SceneAPI, SceneFilter, get_filtered_scenes

scene_filter = SceneFilter(
    # datasets=["nuplan-mini"],
    # datasets=["nuscenes-interpolated-mini"],
    datasets=["carla"],
    split_names=None,
    log_names=None,
    # scene_uuids=[
    #     "28a0f85d-2eaf-5e76-966b-6c2dc1dcb11e",
    #     "cdb256f4-4be5-56df-8b3b-447a49b00f2d",
    #     "2685645c-2a31-552b-a1cf-9dd1eb6de030",
    #     "6586443b-a05c-55f1-8513-bd8a78436889",
    #     "7ff3b3a2-555f-59aa-9f49-36823b7e308f",
    #     "184c7ef9-2452-58b8-99c8-574f196544ed",
    # ],
    # target_iteration_duration_s=0.1,  # 10Hz iteration frequency
    future_duration_s=0.1,  # Look up to 1 second into the future.
    history_duration_s=0.0,  # Look up to 0.5 seconds into the past.
    # timestamp_threshold_s=0.05,  # Allow for up to 50ms timestamp misalignment between modalities.
    # required_scene_modalities=["ego_state_se3", "lidar.lidar_merged"],
    shuffle=False,
)
scenes = get_filtered_scenes(scene_filter)

dataset_splits = set(scene.log_metadata.split for scene in scenes)
print(f"Found {len(scenes)} scenes from {len(dataset_splits)} datasplits:")
for split in dataset_splits:
    print(f" - {split}")

In [ ]:
from py123d.api.scene.arrow.arrow_scene_api import ArrowSceneAPI

from nav123d.agents.base_agent import BaseAgent
from nav123d.agents.constant_velocity_agent import ConstantVelocityAgent
from nav123d.agents.ego_status_mlp_agent import EgoStatusMLPAgent
from nav123d.agents.log_replay_agent import LogReplayAgent
from nav123d.agents.transfuser.transfuser_agent import TransfuserAgent
from nav123d.agents.transfuser.transfuser_config import TransfuserConfig
from nav123d.api.arrow_agent_api import ArrowOracleAgentAPI, ArrowPlannerAgentAPI, ArrowSensorAgentAPI
from nav123d.api.base_agent_api import AgentAPI, ObservationType

ego_mlp_seed = 0
cv_agent = ConstantVelocityAgent()
log_replay_agent = LogReplayAgent()
ego_status_agent = EgoStatusMLPAgent(
    checkpoint_path=f"/home/daniel/Downloads/ego_status_mlp_seed_{ego_mlp_seed}.ckpt", hidden_layer_dim=512, lr=1e-4
)


transfuser_seed = 0
transfuser_agent = TransfuserAgent(
    checkpoint_path=f"/home/daniel/Downloads/transfuser_seed_{transfuser_seed}.ckpt",
    config=TransfuserConfig(latent=False),
    lr=1e-4,
)


ltf_seed = 0
ltf_agent = TransfuserAgent(
    checkpoint_path=f"/home/daniel/Downloads/ltf_seed_{ltf_seed}.ckpt",
    config=TransfuserConfig(latent=True),
    lr=1e-4,
)

agents: dict[str, BaseAgent] = {
    "cv": cv_agent,
    "lr": log_replay_agent,
    "es": ego_status_agent,
    "tf": transfuser_agent,
    "ltf": ltf_agent,
}


def arrow_scene_api_to_agent_api(scene_api: SceneAPI, observation_type: ObservationType) -> AgentAPI:
    """Helper function to convert from SceneAPI to AgentAPI for agent trajectory computation."""
    assert isinstance(scene_api, ArrowSceneAPI), "scene_api should be of type ArrowSceneAPI!"
    log_dir = scene_api._log_dir
    scene_metadata = scene_api._scene_metadata
    api_init = {
        observation_type.SENSOR: ArrowSensorAgentAPI,
        observation_type.PLANNER: ArrowPlannerAgentAPI,
        observation_type.ORACLE: ArrowOracleAgentAPI,
    }
    return api_init[observation_type](log_dir=log_dir, scene_metadata=scene_metadata)

In [ ]:
import matplotlib.pyplot as plt
from py123d.visualization.matplotlib.observation import add_scene_on_ax

from nav123d.geometry.trajectory import TrajectorySampling, TrajectorySE2
from nav123d.metrics.pdm_metric import _resample_trajectory_se2  # noqa: PLC2701

# scene: SceneAPI = np.random.choice(scenes)  # type: ignore
scene = scenes[10]

agent_name = "ltf"

fig, ax = plt.subplots(figsize=(10, 10))
add_scene_on_ax(ax, scene, radius=40)

agent = agents[agent_name]

agent.initialize()

trajectory = agent.compute_trajectory(agent_api=scene)  # type: ignore

assert isinstance(trajectory, TrajectorySE2), "Trajectory should be of type TrajectorySE2 for plotting!"

ego_state_se2 = scene.get_ego_state_se3_at_iteration(0).ego_state_se2  # type: ignore
trajectory_abs = _resample_trajectory_se2(
    trajectory=trajectory,
    sampling=TrajectorySampling(time_horizon=4, interval_length=0.1),
    initial_ego_state_se2=ego_state_se2,
    convert_to_absolute=True,
)
ax.plot(
    trajectory_abs.pose_se2_array[:, 0],
    trajectory_abs.pose_se2_array[:, 1],
    # marker="o",
    # markersize=3,
    linewidth=2,
    color="red",
    label="Predicted Trajectory",
    zorder=5,
)